# Skenario Deteksi Tumor Payudara Menggunakan Random Forest

Notebook ini digunakan untuk melatih Model Machine Learning (`RandomForestClassifier`) yang mengklasifikasikan skenario tumor payudara berdasarkan data frekuensi (Frequency) dan dB dari file CSV hasil simulasi.

## 🛠️ Analisis Kesalahan Model Sebelumnya
Sebelumnya, terdapat kesalahan pencocokan label dalam pencarian substring pada folder data:
1. **Substring Overlap**: 
   - `'i dan ii'` mencocokkan `'i dan iii'` (karena `ii` adalah prefix dari `iii`). Hal ini menyebabkan data `'kuadran_I_III'` salah terlabeli sebagai `'kuadran_I_II'`.
   - `'ii dan iv'` mencocokkan `'iii dan iv'` (karena `ii` adalah substring dari `iii`). Hal ini menyebabkan data `'kuadran_III_IV'` salah terlabeli sebagai `'kuadran_II_IV'`.
2. **Kesalahan Penulisan Spasi/Underscore**: File di folder `upload/` seperti `simulasi kuadran I.csv` ditulis dengan spasi, sementara pengecekan substring mencari `kuadran_i.csv` (dengan underscore). Akibatnya file tersebut salah terlabeli sebagai `tanpa_tumor`.

**Solusi**: Kami memperbaiki fungsi pelabelan dengan menggunakan *regex word tokenization* (`re.findall(r'[a-zA-Z0-9]+', path)`) untuk memisahkan setiap karakter Roman numeral secara terpisah (`i`, `ii`, `iii`, `iv`).


## 1. Import Library
Pertama, kita import library Python yang diperlukan untuk pemrosesan data, pemodelan, dan evaluasi hasil.

In [ ]:
import os
import re
import csv
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

print("Library berhasil di-load!")

## 2. Definisi Parameter dan Kelas
Kita tentukan frekuensi target (30 titik dari 1.5 GHz hingga 4.4 GHz) dan 12 kelas klasifikasi.

In [ ]:
TARGET_FREQS = np.linspace(1.5, 4.4, 30)

CLASSES = [
    'tanpa_tumor', 'dengan_tumor', 
    'kuadran_I', 'kuadran_II', 'kuadran_III', 'kuadran_IV',
    'kuadran_I_II', 'kuadran_I_III', 'kuadran_I_IV', 
    'kuadran_II_III', 'kuadran_II_IV', 'kuadran_III_IV'
]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASSES)}
print("Parameter dan kelas berhasil didefinisikan.")

## 3. Parser CSV dan Fungsi Pelabelan
- `parse_csv_file`: membaca file CSV dan mengekstrak kolom frekuensi dan dB.
- `label_from_path`: melabeli data berdasarkan nama folder dan filenya secara konsisten menggunakan tokenisasi kata reguler.

In [ ]:
def parse_csv_file(filepath):
    freqs = []
    dbs = []
    
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()
        lines = content.strip().split('\n')
        if not lines:
            return None
        
        first_line = lines[0].split(',')
        has_header = False
        try:
            float(first_line[0].replace('"', '').strip())
        except ValueError:
            has_header = True
            
        start_idx = 1 if has_header else 0
        
        for line in lines[start_idx:]:
            row = line.split(',')
            if len(row) < 2:
                continue
            try:
                freq = float(row[0].replace('"', '').strip())
                db = float(row[1].replace('"', '').strip())
                freqs.append(freq)
                dbs.append(db)
            except ValueError:
                pass
                
    if len(freqs) == 0:
        return None
        
    return np.array(freqs), np.array(dbs)

def label_from_path(root, filename):
    path_norm = os.path.join(root, filename).replace('\\', '/').lower()
    
    # 1. Baseline / Tanpa Tumor check
    if 'tanpa' in path_norm:
        return 'tanpa_tumor'
        
    # 2. 2 Tumor (Double Quadrant) check
    if '2 tumor' in path_norm or '2_tumor' in path_norm or '2tumor' in path_norm:
        words = re.findall(r'[a-zA-Z0-9]+', path_norm)
        has_i = 'i' in words
        has_ii = 'ii' in words
        has_iii = 'iii' in words
        has_iv = 'iv' in words
        
        if has_i and has_ii:
            return 'kuadran_I_II'
        elif has_i and has_iii:
            return 'kuadran_I_III'
        elif has_i and has_iv:
            return 'kuadran_I_IV'
        elif has_ii and has_iii:
            return 'kuadran_II_III'
        elif has_ii and has_iv:
            return 'kuadran_II_IV'
        elif has_iii and has_iv:
            return 'kuadran_III_IV'
            
    # 3. Tumor di tengah / Dengan Tumor check
    if 'tumor di tengah' in path_norm or 'dengan_tumor' in path_norm or 'dengan tumor' in path_norm:
        return 'dengan_tumor'
        
    # 4. Single Quadrant check
    words = re.findall(r'[a-zA-Z0-9]+', path_norm)
    if 'kuadran' in words or 'kuadaran' in words:
        if 'iv' in words:
            return 'kuadran_IV'
        elif 'iii' in words:
            return 'kuadran_III'
        elif 'ii' in words:
            return 'kuadran_II'
        elif 'i' in words:
            return 'kuadran_I'
            
    return 'tanpa_tumor'

## 4. Loading Dataset dan Preprocessing
Kita memindai folder `Data/` dan `upload/` untuk melabeli, parsing CSV, dan melakukan interpolasi linier ke frekuensi target.

In [ ]:
X = []
y = []

print("Melakukan preprocessing data...")

# 1. Membaca dataset dari folder Data
for root, dirs, files in os.walk('Data'):
    for file in files:
        if file.endswith('.csv'):
            filepath = os.path.join(root, file)
            lbl = label_from_path(root, file)
            if lbl not in CLASS_TO_IDX:
                continue
            
            parsed = parse_csv_file(filepath)
            if parsed is None:
                continue
            
            freqs, dbs = parsed
            # Interpolasi linier frekuensi ke target 30 titik
            db_interp = np.interp(TARGET_FREQS, freqs, dbs)
            
            X.append(db_interp)
            y.append(CLASS_TO_IDX[lbl])

# 2. Membaca data yang diupload
import glob
for filepath in glob.glob('upload/*.csv'):
    filename = os.path.basename(filepath)
    if 'simulasi' in filename.lower():
        lbl = label_from_path('upload', filename)
        if lbl in CLASS_TO_IDX:
            parsed = parse_csv_file(filepath)
            if parsed is not None:
                freqs, dbs = parsed
                db_interp = np.interp(TARGET_FREQS, freqs, dbs)
                # Gandakan sampel file upload agar memiliki pengaruh tinggi saat pemodelan
                for _ in range(20):
                    X.append(db_interp)
                    y.append(CLASS_TO_IDX[lbl])

X = np.array(X)
y = np.array(y)

print(f"Data berhasil diproses! Shape X: {X.shape}, Shape y: {y.shape}")

## 5. Cek Keseimbangan Kelas
Mari kita verifikasi sebaran data pada masing-masing kelas.

In [ ]:
for idx, name in enumerate(CLASSES):
    count = np.sum(y == idx)
    print(f"Kelas {idx:02d} ({name:<15}): {count} sampel")

## 6. Latih Model Random Forest
Kita melatih `RandomForestClassifier` dengan 100 pohon (`n_estimators=100`) dan kedalaman penuh untuk kecocokan maksimal.

In [ ]:
print("Melatih model Random Forest...")
model = RandomForestClassifier(n_estimators=100, max_depth=None, min_samples_split=2, random_state=42)
model.fit(X, y)

train_acc = model.score(X, y)
print(f"Akurasi Model pada Data Training: {train_acc * 100:.2f}%")

## 7. Evaluasi Model
Mari kita lihat laporan klasifikasi detail dan matriks kebingungan (confusion matrix).

In [ ]:
y_pred = model.predict(X)
print("=== Laporan Klasifikasi ===")
print(classification_report(y, y_pred, target_names=CLASSES, zero_division=0))

## 8. Simpan Model Hasil Latihan ke Joblib
Modul API Next.js akan memuat model dari folder `src/lib/tumor_model.joblib`. Kita simpan model kesana.

In [ ]:
os.makedirs('src/lib', exist_ok=True)
model_path = os.path.join('src', 'lib', 'tumor_model.joblib')
joblib.dump(model, model_path)
print(f"Model berhasil disimpan secara otomatis ke: {os.path.abspath(model_path)}")